### Ollama DocChat GenAI App Using LangChain

In [ ]:
import os
from dotenv import load_dotenv

# Environment Variables
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") 
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# Data Ingestion
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
docs = loader.load()
docs

C:\Users\viren\AppData\Local\Temp\ipykernel_21612\2430595511.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\viren\anaconda3\envs\langchain_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewEngineTraceDebugObserveQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdvanced tracing techniquesData & privacyTroubleshooting guidesReferenceOverviewLangSmith Python SDKLangSmith JS/TS SDKLangSmith Go SDKLa

In [ ]:
# Data Transformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewEngineTraceDebugObserveQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdvanced tracing techniquesData & privacyTroubleshooting guidesReferenceOverviewLangSmith Python SDKLangSmith JS/TS SDKLangSmith Go SDKLa

In [4]:
# Embeddings 
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='qwen3-embedding:4b')

C:\Users\viren\AppData\Local\Temp\ipykernel_21612\3589562053.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model='qwen3-embedding:4b')


In [ ]:
# VectorStoreDB
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(documents, embeddings)
vector_db.save_local("faiss_index")
vector_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

In [ ]:
# Retriever
retriever = vector_db.as_retriever()

In [ ]:
# Ollama LLM
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:8b")

In [ ]:
# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert AI assistant. 
     Answer the question based ONLY on the provided context. Context: {context}"""),
    ("user", "{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are an expert AI assistant. \n     Answer the question based ONLY on the provided context. Context: {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [ ]:
# Output Parser 
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [ ]:
# LCEL Chain
from langchain_core.runnables import RunnablePassthrough

chain = ( {"context": retriever, "input": RunnablePassthrough()} | prompt | model | output_parser)

In [11]:
# Query
input_query = "Summarize the document"
response = chain.invoke(input_query)
clean_response = response.replace("**", "").replace("\\n", "\n").strip()
print(clean_response)

The document provides a tutorial on integrating LangSmith observability into an LLM application to monitor and trace its behavior across development, testing, and production. Key points include:

1. Setup & Tracing:  
   - Wrap LLM clients (e.g., OpenAI) using `wrap_openai` and annotate functions with `@traceable` to track interactions.  
   - Example: A `retriever` function fetches documents for a customer support bot, and `support_bot` uses these to generate responses.  

2. Pipeline Tracing:  
   - Trace both LLM calls and retrieval steps (e.g., querying a document database) to gain end-to-end visibility into the application's workflow.  

3. Example Use Case:  
   - A support bot answers questions like *"How many users can I have on the Starter plan?"* by retrieving relevant docs (e.g., "Starter plans are limited to 5 users").  

4. Observability Tools:  
   - Use the LangSmith CLI (`langsmith trace list`) or web UI to inspect traces, monitor performance, and debug issues.  

5. St